# 🔮 Predictive Analytics Using Historical Data
**Internship Project | Data Analytics | Thiranex**

---
### 📌 Objective
Build a predictive model to forecast future revenue trends using:
- **Data Cleaning & Preprocessing**
- **Exploratory Data Analysis (EDA)**
- **Linear Regression Model**
- **Time Series Forecasting**
- **Model Evaluation (R², MAE, RMSE)**
- **Future Revenue Predictions**


## ⚙️ Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully!')

## 📂 Step 2 — Load Dataset

In [ ]:
df = pd.read_csv('data/predictive_data.csv', parse_dates=['Date'])

print(f'✅ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'📅 Date Range: {df.Date.min().date()} to {df.Date.max().date()}')
df.head(10)

## 🔍 Step 3 — Data Cleaning & Preprocessing

In [ ]:
print('📋 Dataset Info:')
print('-'*45)
print(df.dtypes)
print(f'\n🔎 Missing Values:')
print(df.isnull().sum())
print(f'\n📊 Duplicates: {df.duplicated().sum()}')
print(f'\n✅ Data Quality: CLEAN - No missing values or duplicates!')

In [ ]:
# Feature Engineering
df['Year']        = df['Date'].dt.year
df['Month']       = df['Date'].dt.month
df['Week']        = df['Date'].dt.isocalendar().week.astype(int)
df['Quarter']     = df['Date'].dt.quarter
df['Month_Name']  = df['Date'].dt.strftime('%b %Y')
df['Day_of_Week'] = df['Date'].dt.dayofweek
df['Month_Num']   = (df['Year'] - df['Year'].min()) * 12 + df['Month']

# Encode categorical columns
le = LabelEncoder()
df['Region_Enc']   = le.fit_transform(df['Region'])
df['Category_Enc'] = le.fit_transform(df['Category'])
df['Season_Enc']   = le.fit_transform(df['Season'])

print('✅ Feature Engineering Complete!')
print(f'New columns added: Year, Month, Week, Quarter, Month_Num, encodings')
df.head()

## 📊 Step 4 — Exploratory Data Analysis (EDA)

In [ ]:
# Summary Statistics
summary = df[['Units_Sold','Marketing_Spend','Discount_Percent','Total_Revenue']].describe()
display(summary.style
    .background_gradient(cmap='Blues')
    .format('{:.2f}')
    .set_caption('📊 Statistical Summary'))

In [ ]:
# Monthly Revenue Trend
monthly = df.groupby('Month_Name')['Total_Revenue'].sum().reset_index()
monthly['Date_Sort'] = pd.to_datetime(monthly['Month_Name'], format='%b %Y')
monthly = monthly.sort_values('Date_Sort')

fig1 = px.area(monthly, x='Month_Name', y='Total_Revenue',
               title='📈 Monthly Revenue Trend (2023-2024)',
               labels={'Total_Revenue':'Revenue (₹)','Month_Name':'Month'},
               color_discrete_sequence=['#6366f1'])
fig1.update_traces(fill='tozeroy', line_color='#6366f1', line_width=2.5)
fig1.update_layout(paper_bgcolor='white', plot_bgcolor='#f8fafc',
                   title_font_size=17, height=400,
                   xaxis=dict(tickangle=45, showgrid=False),
                   yaxis=dict(gridcolor='#e2e8f0'))
fig1.show()

In [ ]:
# Revenue by Category and Season
fig2 = make_subplots(rows=1, cols=2,
    subplot_titles=('Revenue by Category', 'Revenue by Season'),
    specs=[[{'type':'pie'},{'type':'pie'}]])

cat_rev = df.groupby('Category')['Total_Revenue'].sum().reset_index()
fig2.add_trace(go.Pie(labels=cat_rev['Category'], values=cat_rev['Total_Revenue'],
    hole=0.4, marker_colors=['#6366f1','#f59e0b','#10b981']), row=1, col=1)

sea_rev = df.groupby('Season')['Total_Revenue'].sum().reset_index()
fig2.add_trace(go.Pie(labels=sea_rev['Season'], values=sea_rev['Total_Revenue'],
    hole=0.4, marker_colors=['#f43f5e','#3b82f6','#84cc16','#f97316']), row=1, col=2)

fig2.update_layout(height=400, paper_bgcolor='white',
    title_text='Revenue Distribution by Category & Season',
    title_font_size=17)
fig2.show()

In [ ]:
# Marketing Spend vs Revenue Correlation
fig3 = px.scatter(df, x='Marketing_Spend', y='Total_Revenue',
    color='Category', size='Units_Sold',
    trendline='ols',
    title='🔗 Marketing Spend vs Revenue (Correlation Analysis)',
    labels={'Marketing_Spend':'Marketing Spend (₹)','Total_Revenue':'Revenue (₹)'},
    color_discrete_sequence=px.colors.qualitative.Bold)
fig3.update_layout(paper_bgcolor='white', plot_bgcolor='#f8fafc',
    title_font_size=17, height=450)
fig3.show()

## 🤖 Step 5 — Linear Regression Model

In [ ]:
# Features and Target
features = ['Month_Num','Units_Sold','Marketing_Spend','Discount_Percent',
            'Unit_Price','Is_Holiday','Region_Enc','Category_Enc','Season_Enc','Quarter']
target   = 'Total_Revenue'

X = df[features]
y = df[target]

# Train Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Metrics
r2_lr   = r2_score(y_test, y_pred_lr)
mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))

print(f'✅ Linear Regression Trained!')
print(f'   R² Score : {r2_lr:.4f} ({r2_lr*100:.1f}% variance explained)')
print(f'   MAE      : ₹{mae_lr:,.0f}')
print(f'   RMSE     : ₹{rmse_lr:,.0f}')

## 🌲 Step 6 — Random Forest Model

In [ ]:
# Train Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# Metrics
r2_rf   = r2_score(y_test, y_pred_rf)
mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print(f'✅ Random Forest Trained!')
print(f'   R² Score : {r2_rf:.4f} ({r2_rf*100:.1f}% variance explained)')
print(f'   MAE      : ₹{mae_rf:,.0f}')
print(f'   RMSE     : ₹{rmse_rf:,.0f}')

## 📊 Step 7 — Model Evaluation & Comparison

In [ ]:
# Model Comparison Table
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'R² Score': [round(r2_lr,4), round(r2_rf,4)],
    'MAE (₹)': [round(mae_lr,0), round(mae_rf,0)],
    'RMSE (₹)': [round(rmse_lr,0), round(rmse_rf,0)],
    'Accuracy (%)': [round(r2_lr*100,1), round(r2_rf*100,1)]
})

display(comparison.style
    .background_gradient(subset=['R² Score','Accuracy (%)'], cmap='Greens')
    .background_gradient(subset=['MAE (₹)','RMSE (₹)'], cmap='Reds_r')
    .format({'MAE (₹)':'₹{:,.0f}','RMSE (₹)':'₹{:,.0f}','Accuracy (%)':'{:.1f}%'})
    .set_caption('🏆 Model Comparison')
    .hide(axis='index'))

In [ ]:
# Actual vs Predicted Chart
fig4 = go.Figure()
x_axis = list(range(len(y_test)))

fig4.add_trace(go.Scatter(x=x_axis, y=y_test.values,
    mode='lines+markers', name='Actual Revenue',
    line=dict(color='#6366f1', width=2),
    marker=dict(size=6)))

fig4.add_trace(go.Scatter(x=x_axis, y=y_pred_lr,
    mode='lines+markers', name='Linear Regression',
    line=dict(color='#f59e0b', width=2, dash='dash'),
    marker=dict(size=6)))

fig4.add_trace(go.Scatter(x=x_axis, y=y_pred_rf,
    mode='lines+markers', name='Random Forest',
    line=dict(color='#10b981', width=2, dash='dot'),
    marker=dict(size=6)))

fig4.update_layout(
    title='📊 Actual vs Predicted Revenue — Model Comparison',
    xaxis_title='Test Samples',
    yaxis_title='Revenue (₹)',
    paper_bgcolor='white', plot_bgcolor='#f8fafc',
    title_font_size=17, height=450,
    legend=dict(x=0.01, y=0.99),
    yaxis=dict(gridcolor='#e2e8f0')
)
fig4.show()

In [ ]:
# Feature Importance
feat_imp = pd.DataFrame({
    'Feature': features,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=True)

fig5 = px.bar(feat_imp, x='Importance', y='Feature',
    orientation='h',
    title='🔍 Feature Importance (Random Forest)',
    color='Importance', color_continuous_scale='Blues')
fig5.update_layout(paper_bgcolor='white', plot_bgcolor='#f8fafc',
    title_font_size=17, height=420, coloraxis_showscale=False)
fig5.show()

## 🔮 Step 8 — Time Series Forecasting (Next 3 Months)

In [ ]:
# Monthly aggregation for time series
ts = df.groupby('Month_Num')['Total_Revenue'].sum().reset_index()

# Fit trend line
X_ts = ts['Month_Num'].values.reshape(-1,1)
y_ts = ts['Total_Revenue'].values
lr_ts = LinearRegression()
lr_ts.fit(X_ts, y_ts)

# Predict next 3 months
last_month = ts['Month_Num'].max()
future_months = np.array([last_month+1, last_month+2, last_month+3]).reshape(-1,1)
future_preds  = lr_ts.predict(future_months)

# Dates for future
future_dates = pd.date_range(start='2025-01-01', periods=3, freq='MS')

print('🔮 Future Revenue Forecast:')
for i, (d, p) in enumerate(zip(future_dates, future_preds)):
    print(f'   {d.strftime("%B %Y")}: ₹{p:,.0f}')

In [ ]:
# Time Series Forecast Chart
hist_dates = pd.date_range(start='2023-01-01', periods=len(ts), freq='2W')

fig6 = go.Figure()

# Historical
fig6.add_trace(go.Scatter(
    x=hist_dates, y=y_ts,
    mode='lines+markers', name='Historical Revenue',
    line=dict(color='#6366f1', width=2.5),
    marker=dict(size=5)))

# Trend line
trend_pred = lr_ts.predict(X_ts)
fig6.add_trace(go.Scatter(
    x=hist_dates, y=trend_pred,
    mode='lines', name='Trend Line',
    line=dict(color='#f59e0b', width=1.5, dash='dash')))

# Future forecast
fig6.add_trace(go.Scatter(
    x=future_dates, y=future_preds,
    mode='lines+markers', name='Forecast (Next 3 Months)',
    line=dict(color='#10b981', width=2.5, dash='dot'),
    marker=dict(size=10, symbol='star')))

# Confidence band
margin = future_preds * 0.08
fig6.add_trace(go.Scatter(
    x=list(future_dates)+list(future_dates[::-1]),
    y=list(future_preds+margin)+list((future_preds-margin)[::-1]),
    fill='toself', fillcolor='rgba(16,185,129,0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='Confidence Interval'))

fig6.update_layout(
    title='🔮 Revenue Time Series Forecast — Next 3 Months',
    xaxis_title='Date', yaxis_title='Revenue (₹)',
    paper_bgcolor='white', plot_bgcolor='#f8fafc',
    title_font_size=17, height=480,
    yaxis=dict(gridcolor='#e2e8f0'),
    legend=dict(x=0.01, y=0.99)
)
fig6.show()

## 📋 Step 9 — Quarterly Revenue Analysis

In [ ]:
qtr = df.groupby(['Year','Quarter']).agg(
    Revenue=('Total_Revenue','sum'),
    Orders=('Units_Sold','sum'),
    Avg_Marketing=('Marketing_Spend','mean')
).reset_index()
qtr['Period'] = 'Q' + qtr['Quarter'].astype(str) + '-' + qtr['Year'].astype(str)

fig7 = make_subplots(rows=1, cols=2,
    subplot_titles=('Quarterly Revenue', 'Quarterly Units Sold'))

colors = ['#6366f1','#f59e0b','#10b981','#f43f5e',
          '#3b82f6','#84cc16','#f97316','#8b5cf6']

fig7.add_trace(go.Bar(x=qtr['Period'], y=qtr['Revenue'],
    marker_color=colors, name='Revenue',
    text=qtr['Revenue'].apply(lambda x: f'₹{x/1e6:.1f}M'),
    textposition='outside'), row=1, col=1)

fig7.add_trace(go.Bar(x=qtr['Period'], y=qtr['Orders'],
    marker_color=colors, name='Units',
    text=qtr['Orders'], textposition='outside'), row=1, col=2)

fig7.update_layout(height=430, paper_bgcolor='white',
    plot_bgcolor='#f8fafc', showlegend=False,
    title_text='Quarterly Performance Analysis',
    title_font_size=17)
fig7.show()

## 💡 Step 10 — Business Insights & Recommendations

In [ ]:
best_model   = 'Random Forest' if r2_rf > r2_lr else 'Linear Regression'
best_r2      = max(r2_rf, r2_lr)
best_cat     = df.groupby('Category')['Total_Revenue'].sum().idxmax()
best_season  = df.groupby('Season')['Total_Revenue'].sum().idxmax()
best_region  = df.groupby('Region')['Total_Revenue'].sum().idxmax()
corr_mkt     = df['Marketing_Spend'].corr(df['Total_Revenue'])

insights_html = f"""
<div style='background:#1e293b;color:white;padding:28px;border-radius:14px;font-family:sans-serif'>
  <h2 style='color:#67e8f9;margin-top:0'>💡 Key Predictive Insights & Recommendations</h2>
  <ul style='line-height:2.4;font-size:15px'>
    <li>🏆 <b>Best Model:</b> <span style='color:#86efac'>{best_model}</span> with <span style='color:#86efac'>{best_r2*100:.1f}% accuracy</span></li>
    <li>📈 <b>Revenue Trend:</b> <span style='color:#86efac'>Consistent upward growth</span> observed across 2023-2024</li>
    <li>🔮 <b>Jan 2025 Forecast:</b> <span style='color:#fbbf24'>₹{future_preds[0]:,.0f}</span> expected revenue</li>
    <li>🔮 <b>Feb 2025 Forecast:</b> <span style='color:#fbbf24'>₹{future_preds[1]:,.0f}</span> expected revenue</li>
    <li>🔮 <b>Mar 2025 Forecast:</b> <span style='color:#fbbf24'>₹{future_preds[2]:,.0f}</span> expected revenue</li>
    <li>📦 <b>Top Category:</b> <span style='color:#86efac'>{best_cat}</span> drives maximum revenue</li>
    <li>🌸 <b>Best Season:</b> <span style='color:#86efac'>{best_season}</span> records peak sales</li>
    <li>🌍 <b>Top Region:</b> <span style='color:#86efac'>{best_region}</span> leads in revenue</li>
    <li>📣 <b>Marketing Impact:</b> <span style='color:#86efac'>{corr_mkt:.2f} correlation</span> between marketing spend and revenue</li>
    <li>💡 <b>Recommendation:</b> Increase marketing budget during <span style='color:#fbbf24'>{best_season} season</span> in <span style='color:#fbbf24'>{best_region} region</span> for maximum ROI</li>
  </ul>
</div>
"""
display(HTML(insights_html))

---
## ✅ Project Complete!
**Predictive Analytics Using Historical Data** | Data Analytics Internship | Thiranex

**Tools Used:** Python, Pandas, Scikit-learn, Plotly, Jupyter Notebook
**Models Built:** Linear Regression, Random Forest Regressor, Time Series Forecasting
**Metrics Used:** R² Score, MAE, RMSE
**Key Output:** 3-Month Revenue Forecast with Confidence Interval